# 04 — 적합도 회귀 입력 특성 절제 실험(ablation) (`results/fitpred_ablation`)

> **GPU 상주 판** — `04-ablation_fitpred.ipynb` 와 같은 규약에서 ESM 토큰까지 모든 입력을 GPU 메모리에 올려 배치를 GPU 안에서 색인으로 만든다 (VRAM 180 GB급, ESM 토큰 bf16 약 131 GiB). 결과 폴더는 같다.

`03-train_fitpred_gpu.ipynb` 와 같은 규약(미니 MLP 1000개, 주 지표 `_pt` · 보조 지표 `_all`)에서 **입력 조합만 바꾼** 13가지 ablation 을 중첩 교차검증으로 평가하고, ablation별 최종 모델과 종합 csv 를 만든다.

| ablation | 입력 | 연결 벡터 폭 | 전체 입력 대비 |
|---|---|---|---|
| `full` (기준) | seq + esm + prot + bio + ctx × 3 | 2712 | — (`results/fitpred`) |
| `noctx` | seq + esm + prot + bio | 2508 | 문맥 특성 3종 제외 |
| `nobio` | seq + esm + prot + ctx × 3 | 2572 | Biopython 특성 제외 |
| `noesm` | seq + prot + bio + ctx × 3 | 1560 | ESM 토큰 제외 |
| `noseq` | esm + prot + bio + ctx × 3 | 2648 | 정수 서열 제외 |
| `noder` | seq + esm + prot | 2368 | 파생 변수(bio · ctx × 3) 제외 |
| `noemb` | seq + bio + ctx × 3 | 408 | ESM 토큰 · ProteomeLM 제외 |
| `seq-prot` | seq + prot | 1216 | 정수 서열 + ProteomeLM만 |
| `esm-prot` | esm + prot | 2304 | 두 임베딩만 |
| `prot` | prot | 1152 | ProteomeLM만 |
| `poolesm` | seq + esm_mean + prot + bio + ctx × 3 | 2712 | ESM 토큰 → 토큰 평균 벡터 |
| `poolesm-noseq` | esm_mean + prot + bio + ctx × 3 | 2648 | 토큰 평균 벡터, 정수 서열 제외 |
| `poolesm-noder` | seq + esm_mean + prot | 2368 | 토큰 평균 벡터, 파생 변수 제외 |
| `poolesm-prot` | esm_mean + prot | 2304 | 토큰 평균 벡터, 두 임베딩만 |

- esm = ESM 토큰 임베딩, seq = 정수 인코딩 서열, esm_mean = ESM 토큰 평균 임베딩, ctx × 3 = `ctx` · `ctxp` · `ctxb`

진행: 1 데이터 → 2 폴드 분할 → 3 모델 · 손실 · 예측 t · 지표 → 4 ablation별 중첩 교차검증 · 최종 모델 → 5 ablation 종합

In [1]:
# !pip install torch scikit-learn h5py pandas matplotlib tqdm biopython
# !pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # 배치 모양이 계속 바뀌어도 VRAM 조각화를 줄인다 (torch 불러오기 전)
import time, numpy as np, pandas as pd, torch
from proteomefit import batching, data, features, folds, cv
from proteomefit.config import Config, setup_torch
from proteomefit.models import ds_blocks, param_summary
from proteomefit.tasks import FitTask

cfg = Config(
    out_dir   = '../results/fitpred_ablation',
    full_dir  = '../results/fitpred_10',   # 전체 입력 모델 결과 (비교 기준, 03 노트북)
    droprate  = 0.1,                         # 연결 벡터 · 미니 MLP 은닉층 · PMA 드롭아웃
    n_branch  = 1000, br_feat = 128,          # 미니 MLP 수 · 가지당 입력 열 수
    br_hidden = (32,),                        # 미니 MLP 은닉층별 노드 수 (길이 = 은닉층 수)
    block_p   = 0.5,                          # 가지가 각 입력 블록을 고를 확률
    bootstrap = True,                         # 가지마다 훈련 샘플을 부트스트랩해 따로 학습 (배깅)
    w_clip    = (0.3, 3.0),                   # 훈련 샘플 가중치 clip(|comb_t|) 범위
    pt_q      = 0.10,                         # 주 지표 _pt 의 예측 t 상위 비율 (테스트 genus 기준)
    lr        = 1e-4,                         # 학습률 (AdamW)
    min_delta = 0.002,                        # 검증 Spearman (|comb_t| >= 2) 개선 인정 하한
)                                             # 나머지 설정 = proteomefit/config.py 기본값

# ablation 이름 -> 입력 (esm = ESM 토큰, seq = 정수 서열, 나머지 = 데이터셋 블록). 적힌 순서대로 연결하며 seq · esm 은 데이터셋 블록보다 앞에 둔다
ABLATIONS = {
    'noctx':         ('seq', 'esm', 'prot', 'bio'),
    'nobio':         ('seq', 'esm', 'prot', 'ctx', 'ctxp', 'ctxb'),
    'noesm':         ('seq', 'prot', 'bio', 'ctx', 'ctxp', 'ctxb'),
    'noseq':         ('esm', 'prot', 'bio', 'ctx', 'ctxp', 'ctxb'),
    'noder':         ('seq', 'esm', 'prot'),
    'noemb':         ('seq', 'bio', 'ctx', 'ctxp', 'ctxb'),
    'seq-prot':      ('seq', 'prot'),
    'esm-prot':      ('esm', 'prot'),
    'prot':          ('prot',),
    'poolesm':       ('seq', 'esm_mean', 'prot', 'bio', 'ctx', 'ctxp', 'ctxb'),
    'poolesm-noseq': ('esm_mean', 'prot', 'bio', 'ctx', 'ctxp', 'ctxb'),
    'poolesm-noder': ('seq', 'esm_mean', 'prot'),
    'poolesm-prot':  ('esm_mean', 'prot'),
}
FULL_INPUTS = ('seq', 'esm', 'prot', 'bio', 'ctx', 'ctxp', 'ctxb')   # 비교 기준(전체 입력) 모델의 입력
NEED = set().union(*ABLATIONS.values())      # 불러올 입력 종류
setup_torch()

print('device:', cfg.device, f'| {cfg.inner_k} inner folds (val genus matched to test extrapolation distance) '
      '| loss per-branch weighted 1-CCC | early stop on val Spearman (|comb_t| >= 2)')
print(f'target: per-organism z-scored {cfg.label_col} | {cfg.n_branch} mini-MLPs (bootstrap per branch: {cfg.bootstrap}) | test metrics: '
      f'top {cfg.pt_q:.0%} predicted t in test genus (_pt, main) + all test samples (_all)')
print('ablations:', ', '.join(ABLATIONS))
print('autocast:', cfg.amp_dtype, '| compile:', cfg.compile, '| all inputs GPU-resident')
if cfg.device != 'cpu':
    p = torch.cuda.get_device_properties(0)
    print(f'gpu: {p.name} | VRAM {p.total_memory / 2**30:.0f} GiB | torch {torch.__version__}')

device: cuda:0 | 5 inner folds (val genus at similarity quantiles (1.0, 0.75, 0.5, 0.25, 0.0)) | loss per-branch weighted 1-CCC | early stop on val Spearman (|comb_t| >= 2)
target: per-organism z-scored fit_mean | 1000 mini-MLPs | test metrics: top 10% predicted t in test genus (_pt, main) + all test samples (_all)
ablations: noctx, nobio, noesm, noseq, noder, noemb, seq-prot, esm-prot, prot, poolesm, poolesm-noseq, poolesm-noder, poolesm-prot
autocast: torch.bfloat16 | compile: True | all inputs GPU-resident
gpu: NVIDIA B200 | VRAM 178 GiB | torch 2.8.0+cu128


## 1. 데이터

### 1.1 가변 길이 입력 — 각각 PMA로 벡터 하나로 축약

| 이름 | 형태 | 출처 | 내용 |
|---|---|---|---|
| ESM 토큰 임베딩 | `[T, 1152]` | `emb_esm_600m.h5` | ESM-C 600M 토큰별 표현 (BOS/EOS 포함) |
| 정수 인코딩 서열 | `[L]` | `seq_aa` | 0 = 패딩, 1 ~ 20 = 아미노산, 21 = 그 외. 앞 `MAX_LEN` 잔기 |

### 1.2 데이터셋 블록

| 블록 | 열 | 내용 |
|---|---|---|
| `esm_mean` | 1152 | ESM 토큰 평균 임베딩, 균주 단위 z-정규화 (`poolesm` 계열에서 ESM 토큰 대신 사용) |
| `prot` | 1152 | ProteomeLM 임베딩, 균주 단위 z-정규화 |
| `bio` | 140 | Biopython 서열 특성 (물리화학 12 · 조성 20 · N말단 20 · C말단 20 · 코돈 빈도 64 · GC 4), 균주 단위 z-정규화 |
| `ctx` · `ctxp` · `ctxb` | 68 × 3 | 문맥 특성 — ESM 평균 · ProteomeLM · 둘을 이은 유사도 공간 |

**문맥 특성** — 질의 샘플마다 46개 균주 각각에서 가장 유사한 단백질 1개를 찾고, **참조 균주**(시험 · 검증 · 자기 genus 밖)의 것만 유사도 상위 $k \in \{1, 3, 5, 10, 20\}$ 개로 요약한다. 내부 폴드마다 다시 만든다.

| 묶음 | 열 | 내용 |
|---|---|---|
| `cons_*` | 9 | 상위 $k$ 평균 코사인, 1위와 3위 · 10위의 격차, 코사인 표준편차, 참조 수 |
| `otf_*` | 25 | 참조 단백질의 적합도 · 필수 여부 요약 (원본 척도) |
| `otr_*` | 12 | 같은 요약, 균주 내 순위 척도 |
| `rbh_*` | 14 | 상호 최적 히트만으로 한 요약 |
| `para_*` | 8 | 자기 게놈 안 유사 단백질의 코사인 요약 |

In [3]:
# 라벨·메타 - 적합도 수치가 있는 샘플만 (n_all 217,468 -> n 177,317)
D    = data.load_samples(cfg.base_dir, cfg.label_col, finite_only=True)
task = FitTask(D, cfg)                                          # 라벨 y = 균주별 z-정규화 적합도 [n] · |comb_t| · 조기 종료 마스크
y    = task.y

gf = pd.Series(D.yb[D.keep]).groupby(D.org)                      # 원본 적합도의 균주별 묶음
print(f'samples:{len(D.lab):,} / {len(D.base):,} (dropped essential {int(np.isneginf(D.yb).sum()):,}, '
      f'unknown {int(np.isnan(D.yb).sum()):,})')
print(f'  raw {cfg.label_col}: per-organism mean {gf.mean().min():+.3f}~{gf.mean().max():+.3f} sd {gf.std().min():.3f}~{gf.std().max():.3f}')
print(f'  label (per-organism z): range {y.min():.2f}~{y.max():.2f} | |comb_t| >= {cfg.t_eval} {int(task.ext.sum()):,}')

/tmp/ipykernel_63881/3498578745.py:16: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  yg   = torch.from_numpy(y).float().to(DEVICE)                 # GPU 라벨 [n]


samples:177,317 / 217,468 (dropped essential 18,037, unknown 22,114)
  raw fit_mean: per-organism mean -0.243~-0.018 sd 0.182~0.695
  label (per-organism z): range -27.29~11.15 | |comb_t| >= 2.0 17,311


In [4]:
# Biopython 특성 - 전 유전자에서 계산 -> 균주별 z-정규화 -> 사용 샘플만 [n, 140]
seq_aa_all = D.base.seq_aa.fillna('').to_numpy()
BIOF  = features.bio_features(seq_aa_all, D.base.seq_dna.fillna('').to_numpy())   # [n_all, 140]
D.BIO = features.zscore_per_org(BIOF, D.org_code_all)[D.keep]
del BIOF

# 정수 인코딩 서열 - 앞 max_len 잔기를 한 배열에 이어 붙인다 (샘플 i = seq_flat[seq_off[i]:seq_off[i+1]], 훈련 때 배열째 GPU 상주)
D.seq_flat, D.seq_off, D.seq_len, raw_len = features.encode_seqs(seq_aa_all[D.keep], cfg.max_len)
print(f'seq residues {len(D.seq_flat):,} (max {int(D.seq_len.max())}, truncated {int((raw_len > cfg.max_len).sum())})')
print('bio', D.BIO.shape)

biopython: 100%|██████████| 217468/217468 [00:37<00:00, 5826.16it/s]


seq residues 60,541,732 (max 4096, truncated 80)
bio (177317, 140)


### 직교상동체 목록

전체 217,468개 단백질 각각에 대해 46개 균주마다 코사인이 가장 높은 단백질의 번호와 값(`ORT`), 자기 게놈 안 상위 20개 코사인을 구한다. 임베딩은 차원별 표준화 후 L2 정규화해 내적 = 코사인으로 만든다. 균주 프로테옴 표현 `ORG_VEC` 도 여기서 구한다.

In [5]:
prot_raw, esm_mean_raw = data.load_embeddings(cfg.prot_h5, cfg.mean_h5, D)        # [n_all, 1152] x 2
D.PROT    = features.zscore_per_org(prot_raw, D.org_code_all)[D.keep]             # 데이터셋 prot 블록 [n, 1152]
D.EMEAN   = features.zscore_per_org(esm_mean_raw, D.org_code_all)[D.keep]         # 데이터셋 esm_mean 블록 [n, 1152]
D.ORG_VEC = pd.DataFrame(prot_raw).groupby(D.org_code_all).mean().to_numpy()      # 균주 프로테옴 표현 [n_org, 1152]

# 직교상동체 목록 3종 (ESM · ProteomeLM · 둘을 이은 공간) - 값 = (sim, idx, para)
D.ORT = features.ortholog_lists(esm_mean_raw, prot_raw, D.org_code_all, D.n_org, max(cfg.topk), cfg.sim_chunk, cfg.device)
del prot_raw, esm_mean_raw
print('ortholog lists:', {k: (v[0].shape, v[2].shape) for k, v in D.ORT.items()})

para both: 100%|██████████| 46/46 [00:00<00:00, 411.70it/s]


ortholog lists: {'esm': ((217468, 46), (217468, 20)), 'prot': ((217468, 46), (217468, 20)), 'both': ((217468, 46), (217468, 20))}


In [6]:
# ESM 토큰별 임베딩 - 'esm' 을 쓰는 ablation 이 있을 때만 GPU 한 배열에 이어 붙인다 (샘플 i = esm[esm_off[i]:esm_off[i+1]], bf16)
D.esm = D.esm_off = D.esm_len = None
if 'esm' in NEED:
    D.esm, D.esm_off, D.esm_len = data.load_esm_tokens(cfg.esm_h5, D, cfg.device, cfg.amp_dtype, cfg.esm_chunk, cfg.gpu_reserve)
    print(f'ESM tokens:{len(D.esm):,} x {D.esm.shape[1]} = {D.esm.nbytes / 2**30:.1f} GiB GPU (bf16) | '
          f'free after load {torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB')

ESM -> GPU: 100%|██████████| 107/107 [03:52<00:00,  2.18s/it]

ESM tokens:60,896,366 x 1152 = 130.7 GiB GPU (bf16) | free after load 46.9 GiB


In [7]:
D.block_dim = features.block_dims(D, cfg.seq_dim)                # 입력별 연결 벡터 폭
W_ABL = max(map(len, ABLATIONS))
for name, inp in {'full': FULL_INPUTS, **ABLATIONS}.items():
    print(f'{name:<{W_ABL}}  width {sum(D.block_dim[b] for b in inp):>5}  = ' + ' + '.join(f'{b}({D.block_dim[b]})' for b in inp))
print(f'organisms:{D.lab.org_id.nunique()} | genus:{D.lab.genus.nunique()}')

full           width  2712  = seq(64) + esm(1152) + prot(1152) + bio(140) + ctx(68) + ctxp(68) + ctxb(68)
noctx          width  2508  = seq(64) + esm(1152) + prot(1152) + bio(140)
nobio          width  2572  = seq(64) + esm(1152) + prot(1152) + ctx(68) + ctxp(68) + ctxb(68)
noesm          width  1560  = seq(64) + prot(1152) + bio(140) + ctx(68) + ctxp(68) + ctxb(68)
noseq          width  2648  = esm(1152) + prot(1152) + bio(140) + ctx(68) + ctxp(68) + ctxb(68)
noder          width  2368  = seq(64) + esm(1152) + prot(1152)
noemb          width   408  = seq(64) + bio(140) + ctx(68) + ctxp(68) + ctxb(68)
seq-prot       width  1216  = seq(64) + prot(1152)
esm-prot       width  2304  = esm(1152) + prot(1152)
prot           width  1152  = prot(1152)
poolesm        width  2712  = seq(64) + esm_mean(1152) + prot(1152) + bio(140) + ctx(68) + ctxp(68) + ctxb(68)
poolesm-noseq  width  2648  = esm_mean(1152) + prot(1152) + bio(140) + ctx(68) + ctxp(68) + ctxb(68)
poolesm-noder  width  2368  = seq(

## 2. 폴드 분할 — 외부 · 내부 모두 genus 단위

$$\text{test}_{k} = G_k, \qquad \text{val}_{k,j} = G_{v_{k,j}}, \qquad \text{train}_{k,j} = \{\, i \mid \mathrm{gid}(i) \notin \{k,\ v_{k,j}\} \,\}$$

- $G_g$ — genus $g$ 의 전 샘플
- 외부 폴드 $k$ — 테스트 genus (28개, genus 단위 리브-원-아웃 교차검증(Leave-One-Out Cross-Validation, LOOCV))
- 내부 폴드 $j$ — 검증 genus $v_{k,j}$ 하나 (`INNER_K` = 3개). 내부 폴드 모델 3개의 테스트 지표 평균이 그 genus 의 성능이다

### 검증 genus 선택 — 시험 genus와 외삽 거리가 비슷한 genus

$$\mathbf{v}_g = \frac{1}{\lvert O_g \rvert} \sum_{o \in O_g} \mathbf{u}_o, \qquad s(k, g) = \cos\big(\mathbf{v}_k - \bar{\mathbf{v}},\ \mathbf{v}_g - \bar{\mathbf{v}}\big)$$

- $\mathbf{u}_o$ — 균주 $o$ 의 ProteomeLM 임베딩 평균 (`ORG_VEC`), $\mathbf{v}_g$ — genus $g$ 균주들의 $\mathbf{u}_o$ 평균 (`GVEC`), $\bar{\mathbf{v}}$ — 전 genus 평균
$$s^{\mathrm{nn}}(g \mid X) = \max_{h \notin X \cup \{g\}} s(g, h), \qquad d(v) = \big\lvert s^{\mathrm{nn}}(v \mid \{k\}) - s^{\mathrm{nn}}(k \mid \varnothing) \big\rvert$$

- $s^{\mathrm{nn}}(g \mid X)$ — genus $g$ 와, $X$ 를 뺀 나머지 genus 중 가장 닮은 것의 유사도 = **외삽 거리** (클수록 가까운 친척이 훈련에 있다, `nn_sim`)
- $s^{\mathrm{nn}}(k \mid \varnothing)$ — 시험 genus 가 풀 외삽, $s^{\mathrm{nn}}(v \mid \{k\})$ — 검증 genus $v$ 가 내부 폴드에서 풀 외삽 (시험 genus 도 훈련에서 빠지므로 제외)
- 후보를 $d(v)$ 오름차순으로 세워 앞의 `INNER_K` 개를 검증 genus 로 쓴다 — 검증 문제의 난이도를 시험 문제에 맞춘다
- 시험 genus 의 최근접 genus 는 후보에서 빼 훈련 · 문맥 참조에 남긴다
- 예: 시험 Pseudomonas 의 $s^{\mathrm{nn}} = 0.39$ → $s^{\mathrm{nn}} = 0.41$ 인 Acidovorax 는 $d = 0.02$ 로 뽑히고, 0.89 인 Klebsiella ($d = 0.50$) 는 뽑히지 않는다
- 검증 genus 중 `|comb_t| ≥ 2` 샘플을 조기 종료에 쓴다
- 후보 = 시험 genus 와 그 최근접 genus 를 뺀 모든 genus

In [8]:
# genus 표현 -> 중심화 코사인 유사도 -> 검증 genus 선택 (후보 = 시험 genus 와 그 최근접 genus 를 뺀 전부)
S = folds.make_splits(D.gen, D.org2genus, D.ORG_VEC, cfg.inner_k, candidate=task.val_candidate)
folds.print_splits(S, D.lab, cfg.inner_k)                        # 분할 요약 + 누출 점검 (warn 이 없으면 정상)

nested CV: 28 outer x 5 inner (genus-level) = 140 trainings / 177,317 samples

fold01 Pseudomonas     : test 45,044(10org) | val Dickeya(+0.39, 10,644) Marinobacter(+0.13, 3,648) Sinorhizobium(-0.02, 5,130) Ralstonia(-0.19, 15,070) Azospirillum(-0.37, 4,178)
fold02 Ralstonia       : test 15,070(4org) | val Cupriavidus(+0.69, 6,373) Desulfovibrio(+0.17, 5,236) Dickeya(-0.10, 10,644) Synechococcus(-0.21, 1,898) Shewanella(-0.50, 13,401)
fold03 Shewanella      : test 13,401(4org) | val Kangiella(+0.61, 1,989) Echinicola(+0.15, 3,950) Bacteroides(-0.05, 4,054) Sphingomonas(-0.23, 3,360) Ralstonia(-0.50, 15,070)
fold04 Dickeya         : test 10,644(3org) | val Klebsiella(+0.76, 4,485) Pedobacter(+0.07, 4,359) Acidovorax(-0.09, 3,924) Dyella(-0.18, 3,611) Caulobacter(-0.41, 3,305)
fold05 Cupriavidus     : test  6,373(1org) | val Ralstonia(+0.69, 15,070) Dechlorosoma(+0.08, 2,550) Sinorhizobium(-0.13, 5,130) Klebsiella(-0.19, 4,485) Shewanella(-0.39, 13,401)
fold06 Burkholderia    : test  5,4

## 3. 모델 · 손실 · 예측 t · 지표

### 구조

```
정수 서열 [B,L] ─── 임베딩 + 위치 ─ PMA ── [B,64] ───┐   (입력에 'seq' 가 있을 때)
ESM 토큰 [B,T,1152] ── PMA ────────────── [B,1152] ─┼─ concat ─ 미니 MLP 1000개 ─ 가지별 예측 [B,1000] ─ 평균 μ · 표준편차 σ   (입력에 'esm' 이 있을 때)
데이터셋 [B,d] ──────────────────────────────────────┘   (입력의 데이터셋 블록만)
```

- PMA(Pooling by Multihead Attention) — 학습 가능한 시드 벡터 1개를 쿼리로 토큰열을 어텐션 가중 평균한다
- 미니 MLP 뱅크 — 가지마다 연결 벡터의 입력 블록(seq · esm · 데이터셋 블록)을 각각 확률 `BLOCK_P` 로 고르고, 고른 블록에 열 `BR_FEAT` 개를 균등 배분해 무작위로 뽑는다. 가지 = `[Linear → GELU → Dropout] × len(BR_HIDDEN) → Linear(1)`, 층별 노드 수 `BR_FEAT → *BR_HIDDEN → 1` (예: `BR_FEAT = 128`, `BR_HIDDEN = [128, 64, 32]` → 128 → 128 → 64 → 32 → 1)
- 예: 가지가 `prot` · `ctx` 를 고르면 두 블록에서 각 64열 → 128열 입력
- ablation 입력에 없는 분기는 만들지 않고, 가지가 고르는 블록도 그 ablation 의 블록뿐이다

### 라벨 — 균주별 z-정규화 적합도

$$y_i = (f_i - \mu_{o_i}) / \sigma_{o_i}$$

- $f_i$ — 원본 `fit_mean`, $\mu_o$ · $\sigma_o$ — 균주 $o$ 의 `fit_mean` 평균 · 표준편차. 예: 평균 $-0.2$ · 표준편차 $0.8$, $f = -1.8$ → $y = -2.0$

### 손실 — 가지별 가중 $1 - \mathrm{CCC}$ 의 평균

$$\mathcal{L} = \frac{1}{K} \sum_{k=1}^{K} \Big[ 1 - \frac{2\,\mathrm{cov}_w(\hat{y}_k, y)}{\mathrm{var}_w(\hat{y}_k) + \mathrm{var}_w(y) + (\bar{\hat{y}}_{k,w} - \bar{y}_w)^2} \Big], \qquad w_i = \frac{\mathrm{clip}(\lvert t_i \rvert, 0.3, 3)}{n_{\mathrm{org}}(g_i)\, n_{\mathrm{smp}}(o_i)}$$

- $\hat{y}_k$ — 가지 $k$ 의 배치 예측, $K$ = `N_BRANCH`. 가지마다 따로 라벨을 맞추므로 가지 간 표준편차가 예측 불확실도가 된다
- 부트스트랩 (`BOOTSTRAP = True`) — 훈련 시작 때 가지 $k$ 마다 훈련 샘플 $m$ 개를 복원 추출해 샘플 $i$ 의 등장 횟수 $c_{ik}$ 를 정하고, 가지 $k$ 의 손실에 가중치 $w_i c_{ik}$ 를 쓴다 (랜덤 포레스트의 배깅). 가지마다 약 $e^{-1} pprox 36.8\%$ 샘플은 한 번도 뽑히지 않는다. 예: $m = 5$, 뽑힌 번호 1 · 1 · 3 · 4 · 5 → $c = (2, 0, 1, 1, 1)$
- $\mathrm{cov}_w$ · $\mathrm{var}_w$ · $\bar{\cdot}_w$ — 배치 안에서 $w_i / \sum w$ 로 구한 가중 공분산 · 분산 · 평균
- $w_i$ — `clip(|comb_t|)` 측정 가중치 × 분류군 역크기 (genus마다 같은 비중), 훈련 샘플 평균 1
- 예: $\hat{y} = 0.5\,y$, $\mathrm{var}(y) = 1$ → $\mathrm{CCC} = 1 / 1.25 = 0.8$, 손실 0.2

### 조기 종료 — 검증 genus 중 `|comb_t| ≥ 2` 샘플의 Spearman

균주마다 $\mu$ 와 $y$ 의 Spearman 을 구해 평균한다. `|comb_t|` 는 훈련 손실에서만 샘플 가중치로 쓴다.

### 예측 t — 가지 간 표준편차에 대한 효과 크기

$$\hat{t}_i = \lvert\mu_i\rvert / \sigma_i$$

- $\mu_i$ · $\sigma_i$ — 미니 MLP 1000개 예측의 평균 · 표준편차. 라벨 없이 계산된다. 예: $\mu = -2.0$, $\sigma = 0.25$ → $\hat{t} = 8$

### 분류 지표 — 예측 평균을 0 기준으로 나눈 낮음(0) / 높음(1)

$$MCC = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

- 실제 = $y > 0$, 예측 = $\mu > 0$ (균주 평균보다 높음). ROC AUC · PR AUC 의 점수는 $\mu$ 이고, 한 클래스뿐이면 NaN
- 예: TP 60 · FN 40 · FP 90 · TN 810 → MCC ≈ 0.42

### 테스트 지표 — 테스트 genus 전 샘플, 균주별로 구해 평균

| 열 | 내용 |
|---|---|
| `pt_cut` | 테스트 genus 전 샘플 예측 t 의 상위 10% 경계값 (90 백분위수) |
| `n_pt` | 예측 t ≥ `pt_cut` 인 샘플 수 |
| `spearman_pt` · `pearson_pt` · `ccc_pt` | 예측 t ≥ `pt_cut` 샘플의 Spearman · Pearson · CCC (**주 지표**, 정렬 기준 `spearman_pt`) |
| `spearman_all` · `pearson_all` · `ccc_all` | 테스트 genus 전 샘플의 Spearman · Pearson · CCC (보조 지표) |
| `mcc` · `f1` · `recall` · `precision` · `accuracy` · `roc_auc` · `pr_auc` · `tp` · `tn` · `fp` · `fn` + `_pt` / `_all` | 같은 두 샘플 집합의 분류 지표 |

`n_org` = 주 지표가 계산된 균주 수.

### 구현 위치

| 내용 | 모듈 |
|---|---|
| 문맥 특성 · 데이터셋 (`ctx_block` · `build_dataset`) | `proteomefit/features.py` |
| 모델 (`PMA` · `SeqBranch` · `FitNet`) | `proteomefit/models.py` |
| 배치 구성 · GPU 상주 배치 공급 · 추론 | `proteomefit/batching.py` |
| 라벨 · 손실 · 샘플 가중치 · 조기 종료 지표 · 평가 산출물 (`FitTask`) | `proteomefit/tasks.py` |
| 지표 · 그림 | `proteomefit/metrics.py` · `proteomefit/plots.py` |
| 훈련 루프 (`train_model`) | `proteomefit/train.py` |
| 중첩 교차검증 · 최종 모델 · 요약 (`run_cv` · `train_final` · `print_summary`) | `proteomefit/cv.py` |

## 4. ablation별 중첩 교차검증 · 최종 모델

ablation마다 `03-train_fitpred_gpu.ipynb` 4 · 5절과 같은 과정을 거친다.

| 경로 (`OUT_DIR/`) | 산출물 |
|---|---|
| `<ablation>/fold##_<genus>/inner##/` | `model.pt` · `history.csv` · `history.png` · `metrics_organ.csv` · `scatter.png` · `confusion.png` · `pred.npz` |
| `<ablation>/fold##_<genus>/` | `metrics_folds.csv` |
| `<ablation>/` | `metrics_total.csv` · `model_final.pt` · `history_final.csv` · `history_final.png` |

`RESUME=True` 면 산출물이 모두 있는 내부 폴드 · 최종 모델은 다시 훈련하지 않는다.

In [14]:
os.makedirs(cfg.out_dir, exist_ok=True)
RES, t_start = {}, time.perf_counter()                           # ablation -> genus별 결과 DataFrame
for name, inputs in ABLATIONS.items():
    _m = task.build_model(inputs, cfg.seed)
    print(f'\n##### ablation {name} | {" + ".join(inputs)} | width {_m.d_in} | params {param_summary(_m)}')
    del _m
    t_abl, out = time.perf_counter(), os.path.join(cfg.out_dir, name)
    RES[name] = cv.run_cv(task, D, S, cfg, inputs, out, name)     # 중첩 교차검증 -> out/metrics_total.csv
    cv.train_final(task, D, S, cfg, inputs, out, RES[name], name)  # 최종 모델 -> out/model_final.pt
    r = RES[name]
    print(f'##### {name} done ({cv.hms(time.perf_counter() - t_abl)}) | genus mean '
          + ' '.join(f'{c} {r[c].mean():.3f}±{r[c].std():.3f}' for c in task.brief_cols))
print(f'\nall ablations done | total elapsed {cv.hms(time.perf_counter() - t_start)}')


##### ablation noctx | seq + esm + prot + bio | width 2508 | params 9.76M
=== noctx fold 1/28  Pseudomonas === test 45,044 samples (10 org)
  inner01 val Dickeya          (sim +0.39) best@21   val_sp=0.354 | pt_cut=1.73 Sp=0.232 MCC=0.156 | Sp_all=0.111 MCC_all=0.081 | cached
  inner02 val Marinobacter     (sim +0.13) best@17   val_sp=0.287 | pt_cut=1.72 Sp=0.196 MCC=0.105 | Sp_all=0.108 MCC_all=0.076 | cached
  inner03 val Sinorhizobium    (sim -0.02) best@28   val_sp=0.077 | pt_cut=1.56 Sp=0.276 MCC=0.217 | Sp_all=0.112 MCC_all=0.091 | cached
  inner04 val Ralstonia        (sim -0.19) best@24   val_sp=0.297 | pt_cut=1.69 Sp=0.265 MCC=0.218 | Sp_all=0.112 MCC_all=0.091 | cached
  inner05 val Azospirillum     (sim -0.37) best@2    val_sp=0.531 | pt_cut=1.12 Sp=0.140 MCC=0.022 | Sp_all=0.093 MCC_all=0.078 | cached
  noctx fold 1/28 Pseudomonas (0:00:00) | best@21.0±10.0 val_sp=0.309±0.162 | pt_cut=1.564±0.256 Sp=0.222±0.056 MCC=0.144±0.083 | Sp_all=0.107±0.008 MCC_all=0.083±0.007

=== 

prot inner01:   0%|          | 0/500 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/variables/functions.py:1481: UserWarning: Dynamo does not know how to trace the builtin `<unknown module>.slice.indices.` This function is either a Python builtin (e.g. _warnings.warn) or a third-party C/C++ Python extension (perhaps created with pybind).
If it is a Python builtin, please file an issue on GitHub so the PyTorch team can add support for it and see the next case for a workaround.
If it is a third-party C/C++ Python extension, please either wrap it into a PyTorch-understood custom operator (see https://pytorch.org/tutorials/advanced/custom_ops_landing_page.html for more details) or, if it is traceable, use `torch.compiler.allow_in_graph`.
  torch._dynamo.utils.warn_once(explanation + "\n" + "\n".join(hints))
prot inner01:  23%|██▎       | 113/500 [00:31<01:46,  3.62it/s, loss=0.7472/0.8351 Sp=0.345/0.327 best=0.325@94 pat=20]


  inner01 val Dinoroseobacter  (sim +0.84) best@94   val_sp=0.325 | pt_cut=2.70 Sp=0.320 MCC=0.351 | Sp_all=0.192 MCC_all=0.162


prot inner02:  13%|█▎        | 65/500 [00:17<01:57,  3.71it/s, loss=0.7612/0.6616 Sp=0.328/0.377 best=0.376@46 pat=20]


  inner02 val Acidovorax       (sim +0.04) best@46   val_sp=0.376 | pt_cut=2.66 Sp=0.384 MCC=0.363 | Sp_all=0.198 MCC_all=0.174


prot inner03:   4%|▍         | 22/500 [00:06<02:14,  3.56it/s, loss=0.7885/0.7548 Sp=0.303/0.285 best=0.308@3 pat=20]


... (1,161 outputs left out)

poolesm-prot inner05:  63%|██████▎   | 317/500 [01:25<00:49,  3.71it/s, loss=0.7332/0.5746 Sp=0.389/0.460 best=0.459@298 pat=20]


  inner05 val Paraburkholderia (sim -0.27) best@298  val_sp=0.459 | pt_cut=1.70 Sp=0.314 MCC=0.362 | Sp_all=0.210 MCC_all=0.196
  poolesm-prot fold 28/28 Synechococcus (0:04:01) | best@139.0±109.8 val_sp=0.346±0.075 | pt_cut=1.635±0.698 Sp=0.263±0.033 MCC=0.303±0.045 | Sp_all=0.193±0.023 MCC_all=0.192±0.025



poolesm-prot final: 100%|██████████| 76/76 [00:21<00:00,  3.58it/s, loss=0.7615/nan Sp=0.325/nan best=-2.000@0 pat=0]


  poolesm-prot final model: 76 epochs -> results/fitpred_ablation/poolesm-prot/model_final.pt
##### poolesm-prot done (1:17:49) | genus mean spearman_pt 0.217±0.096 ccc_pt 0.189±0.070 mcc_pt 0.203±0.119 roc_auc_pt 0.623±0.064 spearman_all 0.093±0.053 ccc_all 0.171±0.056 mcc_all 0.084±0.060 roc_auc_all 0.559±0.046

all ablations done | total elapsed 5:41:42


## 5. ablation 종합

`FULL_DIR/metrics_total.csv`(`03-train_fitpred_gpu.ipynb` 결과)가 있으면 비교 기준 `full` 로 함께 싣는다. 모든 ablation 과 `full` 은 같은 폴드 분할을 쓴다.

| 파일 (`OUT_DIR/`) | 내용 |
|---|---|
| `ablation_summary.csv` | ablation별 1줄 — 입력, 연결 벡터 폭, 지표가 계산된 genus 수, `best_epoch_med` · `best_epoch_mean` · `n_epoch_final`, 지표의 genus 평균(`_sd` = genus 간 표준편차), `d_sp`, `n_genus_worse` |
| `ablation_genus_spearman.csv` | genus × ablation `spearman_pt` |

- `d_sp` — ablation 의 genus 평균 `spearman_pt` − `full` 의 값. 예: `full` 0.40, `noctx` 0.32 → $-0.08$
- `n_genus_worse` — `spearman_pt` 이 `full` 보다 낮은 genus 수

In [15]:
summ = cv.summarize_ablations(task, RES, ABLATIONS, FULL_INPUTS, cfg.full_dir, cfg.out_dir,
                              D.block_dim, S.GENERA, cfg.csv_round)   # ablation_summary.csv · genus x ablation 표

     ablation  width  n_genus  n_epoch_final  spearman_pt  ccc_pt  mcc_pt  roc_auc_pt  spearman_all  ccc_all  mcc_all  roc_auc_all  spearman_pt_sd   d_sp  n_genus_worse
         full   2712       28             12        0.310   0.306   0.284       0.659         0.116    0.263    0.115        0.575           0.089  0.000              0
        noctx   2508       28             16        0.224   0.197   0.169       0.615         0.096    0.189    0.088        0.564           0.087 -0.086             25
        nobio   2572       28             11        0.308   0.303   0.270       0.656         0.116    0.264    0.114        0.576           0.092 -0.001             15
        noesm   1560       28             23        0.311   0.292   0.269       0.658         0.116    0.263    0.111        0.576           0.101  0.001             15
        noseq   2648       28             17        0.310   0.295   0.273       0.656         0.116    0.265    0.116        0.575           0.088 -0.000  